In [1]:
import random
import copy

GRID = 6

print("=== Drone Delivery Backward Planning ===")

delivery = input("Delivery type (express/normal): ").lower()

goal_x = int(input("Enter goal row (0-5): "))
goal_y = int(input("Enter goal column (0-5): "))

start = (0,0)
goal = (goal_x,goal_y)

grid = [["." for _ in range(GRID)] for _ in range(GRID)]
grid[start[0]][start[1]] = "S"
grid[goal[0]][goal[1]] = "G"

# -------------------------
# random dynamic obstacles
# -------------------------

num_obs = random.randint(2,3)
obstacles=[]

while len(obstacles)<num_obs:

    r=random.randint(0,GRID-1)
    c=random.randint(0,GRID-1)

    if (r,c)!=start and (r,c)!=goal:
        obstacles.append((r,c))

print("\nInitial Grid")
for r in grid:
    print(r)

print("\nObstacle starting positions:",obstacles)

# -------------------------
# obstacle movement
# -------------------------

def obstacle_position_at_time(t):

    pos=[]

    for r,c in obstacles:
        new_r=(r+t)%GRID
        pos.append((new_r,c))

    return pos

# -------------------------
# route generators
# -------------------------

def route_straight():

    path=[start]
    r,c=start

    while c<goal_y:
        c+=1
        path.append((r,c))

    while r<goal_x:
        r+=1
        path.append((r,c))

    return path


def route_down_first():

    path=[start]
    r,c=start

    while r<goal_x:
        r+=1
        path.append((r,c))

    while c<goal_y:
        c+=1
        path.append((r,c))

    return path


def route_boundary():

    path=[start]
    r,c=start

    while r<GRID-1:
        r+=1
        path.append((r,c))

    while c<goal_y:
        c+=1
        path.append((r,c))

    while r>goal_x:
        r-=1
        path.append((r,c))

    return path


def route_zigzag():

    path=[start]
    r,c=start

    while (r,c)!=goal:

        if c<goal_y:
            c+=1
            path.append((r,c))

        if r<goal_x:
            r+=1
            path.append((r,c))

    return path


routes={
"A_straight":route_straight(),
"B_down_first":route_down_first(),
"C_boundary":route_boundary(),
"D_zigzag":route_zigzag()
}

print("\nCandidate Routes:")
for name,path in routes.items():
    print(name,path)

# -------------------------
# backward planning
# -------------------------

valid_routes={}

for name,path in routes.items():

    collision=False

    total_steps=len(path)

    for t in range(total_steps):

        pos=path[t]

        if pos in obstacle_position_at_time(t):

            print("\nCollision predicted on",name,"at",pos,"time",t)

            collision=True
            break

    if not collision:

        cost=len(path)

        if delivery=="express":
            cost-=2

        valid_routes[name]=(path,cost)


# -------------------------
# choose best route
# -------------------------

if not valid_routes:

    print("\nNo safe path found")

else:

    best=min(valid_routes,key=lambda x:valid_routes[x][1])
    best_path=valid_routes[best][0]

    print("\nSelected Route:",best)

    grid_plan=copy.deepcopy(grid)

    # draw arrows backward from goal
    reversed_path=list(reversed(best_path))

    for i in range(len(reversed_path)-1):

        r,c=reversed_path[i]
        pr,pc=reversed_path[i+1]

        if pc>c:
            grid_plan[r][c]="←"
        elif pc<c:
            grid_plan[r][c]="→"
        elif pr>r:
            grid_plan[r][c]="↑"
        elif pr<r:
            grid_plan[r][c]="↓"

    print("\nBackward Planning Grid (Goal → Start)\n")

    for r in grid_plan:
        print(r)

=== Drone Delivery Backward Planning ===


Delivery type (express/normal):  normal
Enter goal row (0-5):  3
Enter goal column (0-5):  2



Initial Grid
['S', '.', '.', '.', '.', '.']
['.', '.', '.', '.', '.', '.']
['.', '.', '.', '.', '.', '.']
['.', '.', 'G', '.', '.', '.']
['.', '.', '.', '.', '.', '.']
['.', '.', '.', '.', '.', '.']

Obstacle starting positions: [(3, 3), (3, 5)]

Candidate Routes:
A_straight [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (3, 2)]
B_down_first [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2)]
C_boundary [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (5, 1), (5, 2), (4, 2), (3, 2)]
D_zigzag [(0, 0), (0, 1), (1, 1), (1, 2), (2, 2), (3, 2)]

Selected Route: A_straight

Backward Planning Grid (Goal → Start)

['S', '→', '→', '.', '.', '.']
['.', '.', '↓', '.', '.', '.']
['.', '.', '↓', '.', '.', '.']
['.', '.', '↓', '.', '.', '.']
['.', '.', '.', '.', '.', '.']
['.', '.', '.', '.', '.', '.']
